# Adding FHIR data to a FHIR server

Your project may involve ingesting data into FHIR format and sending it to a FHIR server. This notebook is a quick example of how this can be achieved with IRIS health. 

This tutorial will assume you have a FHIR server set up with IRIS-health as described in the [FHIR Server quickstart tutorial](../1-setup/1.1-setup-iris-and-fhir.md). If you have set up the FHIR server by different methods, the differences will be where the requests are sent (Server URL, username and password) but the content will be the same. 

If you need to bulk load FHIR data, for example you have a >1GB file and wish to avoid latency or issues with POST requests, skip to the [Bulk Load](#bulk-load) section.  

## Add existing FHIR files to the FHIR server

You can also add a bundle of FHIR resources to a FHIR Server using HTTP requests - the FHIR server will add the resources to the correct place. For an example, we are going to add 5 patient bundles - that is the complete medical history of the patient. 

All the FHIR data used in this project has been synthetically generated by Synthea. The instructions for this are a [separate tutorial](../6-extras/create-synthetic-fhir-data.md), but are super simple - use the command: 

    docker run --rm -v $PWD/output:/output --name synthea-docker intersystemsdc/irisdemo-base-synthea:version-1.3.4 -p 5

Where -p denotes the number of patients to generate and `$PWD/output` is the output directory of the files (a new folder called output in your current working directory).


This has already been done for you, so there are 5 example patient resources in ../data/fhir. Let's work with these.

In [1]:
import os
fhir_data_path = "../data/fhir/"
os.listdir(fhir_data_path)

['hospitalInformation1760013140357.json',
 'Jarred626_Walsh511_ea6f7f29-8553-4456-8fa9-3d629ffcd5f7.json',
 'Krista314_Mosciski958_79fdc511-4556-435d-b922-7a55d0d6e409.json',
 'Kristofer887_Schowalter414_a7b87618-d13a-4f02-8974-010aa35fb759.json',
 'Marshall526_Mosciski958_dbc8773a-10ac-42e5-9c8c-c8d269ed9179.json',
 'Milissa240_Gaylord332_db7fc62f-0f83-484f-bf84-40e85fc26c4d.json',
 'practitionerInformation1760013140357.json']

In [2]:
import json
import requests
from requests.auth import HTTPBasicAuth

server_url = "http://localhost:52773/fhir/r4/"
username = "SuperUser"
password = "SYS"

# Iterate over the files
for file in os.listdir(fhir_data_path):
    # Read the File
    with open(fhir_data_path+file, "r")as f:
        json_data = json.load(f)
        
    res = requests.post( server_url,
                         json=json_data,
                         headers={"Content-Type": "application/fhir+json"}, 
                         auth=HTTPBasicAuth(username, password))
    print(res)
    if res.status_code!=200:
        print(res.json())

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>


Lets just check the patient resources were added: 

In [3]:
endpoint = "Patient"

res = requests.get(server_url+endpoint, headers={"Accept": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))
print(res)
#print(res.json())
data = res.json() 

for entry in data.get("entry", [])[-5:]: ## Get the last 5 entries
    patient = entry.get('resource', {}) ## Get the patient resource
    print(patient.get('id'), patient.get('name'), patient.get('gender')) ## Print the patient info


<Response [200]>
2979 [{'use': 'official', 'family': 'Walsh511', 'given': ['Jarred626']}] male
3363 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Krista314'], 'prefix': ['Mrs.']}, {'use': 'maiden', 'family': 'Frami345', 'given': ['Krista314'], 'prefix': ['Mrs.']}] female
3242 [{'use': 'official', 'family': 'Schowalter414', 'given': ['Kristofer887'], 'prefix': ['Mr.']}] male
3901 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Marshall526'], 'prefix': ['Mr.']}] male
4463 [{'use': 'official', 'family': 'Gaylord332', 'given': ['Milissa240'], 'prefix': ['Ms.']}] female


And there we have it! How to add FHIR resources into a FHIR server running with InterSystems IRIS for Health. I hope you have found this useful! 

## Creating new FHIR data 

Let's say we want to create a new resource regarding some kind of medical data, rather than just use synthetic generated data. Let's do this using Python.

We are going to add FHIR data to the server in two steps, firstly create a valid fhir resource - the Python package [`fhir.resources`](https://github.com/nazrulworld/fhir.resources) is going to help with this. Then sending a POST request to the server using the standard http `requests` library. 

### Step 1: Create valid resources

Firstly, install the dependency if you haven't already (from requirements.txt). This is [fhir.resources](https://github.com/nazrulworld/fhir.resources). There is much more information about its usage on the linked repo than included here, so check it out if you want more information.


In [4]:
# pip install fhir.resources 

You then need to import the type of resource you would like to make. We will start by creating a new patient using a JSON format object:

In [5]:
from fhir.resources.patient import Patient

data = { ## Create a data object that resembles a fhir resource
    "name": [{
        "use": "official",
        "family": "Kent", 
        "given": ["Clark"]  ## As you can have multiple given names (middle names) this needs to be a list item
    }],
    "birthDate":"1965-02-12", "gender":"male"}
superman = Patient.model_validate(data)

In [6]:
print(superman.birthDate)
print(superman.name)

1965-02-12
[HumanName(fhir_comments=None, extension=None, id=None, family='Kent', family__ext=None, given=['Clark'], given__ext=None, period=None, prefix=None, prefix__ext=None, suffix=None, suffix__ext=None, text=None, text__ext=None, use='official', use__ext=None)]


Alternatively we can use a more Pythonic type way:

In [7]:
batman = Patient.model_construct()


batman.name = [{}] # The name part of the patient resource is a bit of a pain! 
batman.id = "500"
batman.name[0].family = "Wayne"
batman.name[0].given = ["Bruce"]
batman.birthDate = "1982-01-15"
batman.gender= "male"

#### Accessing model data

In [8]:
batman_model = batman.model_dump_json() 
print(batman_model,'\n' ,  type(batman_model), '\n\n') ## This returns a JSON string

superman_model = superman.model_dump() ## This returns a json object (or python dict)

print(superman_model, '\n', type(superman_model))

{"resourceType":"Patient","id":"500","name":[{"family":"Wayne","given":["Bruce"]}],"gender":"male","birthDate":"1982-01-15"} 
 <class 'str'> 


{'resourceType': 'Patient', 'name': [{'use': 'official', 'family': 'Kent', 'given': ['Clark']}], 'gender': 'male', 'birthDate': datetime.date(1965, 2, 12)} 
 <class 'dict'>


There are classes for every fhir resource defined in the FHIR specification, so a similar method can be used with any resource you may need.

Below we can create a DocumentReference resource. If you have worked through the main demo in this repository, you may be familiar with this resource type, but it can be used to link any different type of documents. 


In [9]:
from fhir.resources.documentreference import DocumentReference
from datetime import datetime, timezone
note_resource = DocumentReference.model_construct()


with open("../data/clinical_note.txt", "r", encoding="utf-8") as f:
    note = f.read()
    
# Set the note status
note_resource.status = "current"

# Set the date
note_resource.date = datetime(2026, 9, 10, tzinfo=timezone.utc)
# Set the subject reference (patient ID)
note_resource.subject = {"reference": "500"}
# Set the content with the note text as an attachment, encoding the content in hexadecimal
note_resource.content = [{ "attachment":{"contentType": "text/plain; charset=utf-8", "data": note.encode("utf-8").hex()}}]


### Step 2: Adding data to the server

Similar to how we accessed the data with http requests in the [Querying FHIR data with Python](3.2-querying-fhir-data-with-python.ipynb) demo, we can add the FHIR resources we have created to the server using HTTP requests. We do need to add basic authorisation. 

In [10]:
import requests
from requests.auth import HTTPBasicAuth

## Credentials
username = "SuperUser"
password = "SYS"

## FHIR server location
baseURL = "http://localhost:52773/fhir/r4/"
headers ={    "Content-Type": "application/fhir+json"}
endpoint = "Patient"


res = requests.post(baseURL+endpoint, 
                    data=superman.model_dump_json(), ## Give a JSON string as data - giving it a Python dict may cause errors because of the datetime object.
                    headers= headers, ## We are defining the content type (FHIR+JSON) in the header
                    auth=HTTPBasicAuth(username, password))
print(res)

<Response [201]>


In [11]:
## If you have any error message, you can find some more information with 
if res.status_code!=201:
    print(res.json())
else: 
    print("Data successfully added!")


Data successfully added!


Response 201 means it has been successfully added!

In [12]:
res = requests.post(baseURL+endpoint, data= batman.model_dump_json(), headers= headers, auth=HTTPBasicAuth(username, password))

print(res)

<Response [201]>


#### Checking data is added
Lets just query the endpoint to see whether we have added them correctly:

In [13]:
## FHIR server location
baseURL = "http://localhost:52773/fhir/r4/"

get_headers ={    "Accept": "application/fhir+json"}

endpoint = "Patient"


res = requests.get(baseURL+endpoint, headers=get_headers, auth=HTTPBasicAuth(username, password))
print(res)
#print(res.json())
data = res.json() 

for entry in data.get("entry", [])[-5:]: ## Get the last 5 entries
    patient = entry.get('resource', {}) ## Get the patient resource
    print(patient.get('id'), patient.get('name'), patient.get('gender')) ## Print the patient info


<Response [200]>
3242 [{'use': 'official', 'family': 'Schowalter414', 'given': ['Kristofer887'], 'prefix': ['Mr.']}] male
3901 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Marshall526'], 'prefix': ['Mr.']}] male
4463 [{'use': 'official', 'family': 'Gaylord332', 'given': ['Milissa240'], 'prefix': ['Ms.']}] female
4691 [{'use': 'official', 'family': 'Kent', 'given': ['Clark']}] male
4692 [{'family': 'Wayne', 'given': ['Bruce']}] male


So we have added them! Because we have used a POST request, they are automatically assigned an ID number. We could use a put request if we want to keep the ID number we've assigned them.


### Adding Document Reference

Lets do the same with the document reference resource - only thing we need to change is the endpoint and the data:

In [14]:
res = requests.post(baseURL+"DocumentReference", data=note_resource.model_dump_json(),
                   headers={"Content-Type": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))

print(res)
print(res.json())

<Response [400]>
{'resourceType': 'OperationOutcome', 'issue': [{'severity': 'error', 'code': 'invalid', 'diagnostics': '<HSFHIRErr>MalformedRelativeReference', 'details': {'text': "The reference value '500' in property (subject) of Type 'DocumentReference' is malformed"}, 'expression': ['DocumentReference.subject']}]}


Ok, so the fhir.resources doesn't completely validate our data, but the FHIR server rejects bad data! Lets work through these issues: 

In [15]:
for issue in res.json()["issue"]:
    print(issue["details"]["text"])
    print('\n')

The reference value '500' in property (subject) of Type 'DocumentReference' is malformed




We can see the problem here - the patient reference is wrong. Lets fix it and try again: 

In [16]:
note_resource.subject = {"reference": "Patient/3600"}


In [17]:
res = requests.post(baseURL+"DocumentReference", data=note_resource.model_dump_json(),
                   headers={"Content-Type": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))

print(res)


<Response [201]>


In [18]:
res = requests.get(baseURL+"DocumentReference?patient=3600",
                   headers=get_headers, auth=HTTPBasicAuth(username, password))
print(res)

<Response [200]>


In [19]:
resource = res.json()["entry"][0]["resource"]
# print(resource)
print(bytes.fromhex(resource["content"][0]["attachment"]["data"]).decode("utf-8"))

Patient Name: Clark Kent
DOB: [Redacted]
Date of Visit: 09/10/2025
Presenting Complaint:
Patient presents with complaints of persistent chest discomfort and intermittent episodes of spontaneous x-ray vision activation, particularly during emotionally charged situations (e.g., office meetings, romantic encounters, surprise birthday parties).
History of Present Illness:
Mr. Kent reports a recent increase in involuntary activation of his x-ray vision, resulting in several awkward social encounters, including accidentally seeing through three floors of the Daily Planet building. He also notes mild chest tightness after flying through a cumulonimbus cloud at Mach 3, which he describes as “like being hugged by a thunderstorm.”
Past Medical History:

Kryptonian physiology
Solar overexposure syndrome (resolved)
Mild allergic reaction to kryptonite (ongoing)

Medications:

None (immune to most Earth-based pharmaceuticals)

Allergies:

Kryptonite (severe photosensitivity, weakness, existential d

# Bulk Load

To bulk load large datasets into the FHIR Server, you will likely get a faster load. The downside is that the files need to be on the same filesystem as InterSystems IRIS for Health, so you need to copy files into the docker container or deployed server. 
This can be done with `docker cp` (local) or `scp` (SSH access):

In [ ]:
!docker cp fhirdata iris-fhir:/tmp/fhirdata

In [ ]:
!scp -r fhirdata irisowner@123-45-67-890:/tmp/fhirdata

Then load the data with the IRIS class `HS.FHIRServer.Tools.DataLoader.SubmitResourceFiles()` (via the Python Native API): 

In [2]:
import time 
import iris 
st = time.time()

conn_args = {"hostname": "localhost", "port":1972, "namespace":"FHIRSERVER", "username": "SuperUser", "password":"SYS"}

conn  =iris.connect(**conn_args)

irispy = iris.createIRIS(conn) 

status = irispy.classMethodString("HS.FHIRServer.Tools.DataLoader", # IRIS Class 
                                   "SubmitResourceFiles", # Method name
                                   "/home/irisowner/dev/src/extra_fhir_data", # Path to FHIR bundle 
                                   "FHIRServer", # Type of load (Ignore)
                                   "/fhir/r4", # 
                                   1, # Display progress
                                    "^FHIRLoadLog2" # Log progress to a global 
                                 )

print(status)
conn.close()
time_taken = time.time() - st
print(time_taken)


Starting DataLoader with 7 jobs
Assigned ID 363D5AAD-C6FC-11F1-8D58-F2D4D1010E90 to this run

1. resources.ndjson                  0 KB            1 resource     0.761 secs
2. resources.ndjson                  0 KB            1 resource     0.449 secs
3. resources.ndjson                  0 KB            1 resource     0.453 secs
4. resources.ndjson                  0 KB            1 resource     0.007 secs
5. resources.ndjson                  0 KB            1 resource     0.454 secs
6. resources.ndjson                  0 KB            1 resource     0.456 secs
7. resources.ndjson                  0 KB            1 resource     0.427 secs
8. resources.ndjson                  0 KB            1 resource     0.008 secs
9. resources.ndjson                  0 KB            1 resource     0.006 secs
10. resources.ndjson                 0 KB            1 resource     0.008 secs
11. resources.ndjson                 0 KB            1 resource     0.007 secs
12. resources.ndjson                

## Troubleshooting 

You may run into issues with these approaches if you use 